In [ ]:
from kdrag.all import *
from kdrag.smt import *


class Theory:
    def __init__(self, name : str):
        self.name = name
        self.decls = []
        self.defns = {}
        self.axioms = []
        self.theorems = []
    def define(self, name, definition):
    def declare(self, ):
    def axiom(self, formula : smt.BoolRef):
        a = kd.axiom(formula)
        self.axioms.append(a)
        return a
    def prove(self, *args, **kwargs):
        p = kd.prove(*args, **kwargs)
        self.theorems.append(p)
        return p
    def Theorem(self, *args, **kwargs):
        t = kd.Theorem(*args, **kwargs)
        self.theorems.append(t)
        return t



But then what is the point?
I should just do my lean playing to understand wha a bog standard reals would like like. It might be nice to just back m theorems with lean anyway.
leanpy is dead?
grind can probably solve many of the style of problem I'm doing.
https://github.com/kiranandcode/lean.py yeah dead I'd say


specialize should take multiple args and formulas to discharge implies.


Maybe I should get rid of complexity in Calc

Why did i make QForAll  an implicit And instead of curried?


f(..., x)  == Lambda(y, f(y, x))    as akin to lean's . + .  notation. That's a fun idea.


iteration out of formulas an proofs

x,y,z = prove(And(a,b,c))
p, (q,r) = 


pf.exists([term1, term2]) = Proof(Exists([a,b], pf.bodysubst))

What more do I need to be able to completely build explicit proof terms?

& & &  of Proof
| | | of Proof? 
a 
---- 
a | b

Add more variable control with free_in checks.




In [ ]:
class Hyp:

class Var:

def partial_instan(self, *args):
    trail = []
    fm = self.fm
    pos = 0
    while pos < len(args):
        if smt.is_implies(fm):
            pos += 1
            if isinstance()
            if isinstance(arg[pos], Ellispse):
                trail.append(arg[pos])
            


In [ ]:
print_table : dict[smt.FuncDeclRef, Callable] = {

}
prec_table : {
    
}

def printer(e : smt.ExprRef, prec : int) -> str: # indent, other data?
    if isinstance(e, smt.QuantifierRef):

        vs, body = kd.utils.open_binder_unhygienic(e)
        vs_str =
        for v in vs:
            vs_str.append(str(v) + " : " + str(v.sort()))
        vs_str = " ".join(vs_str)
        if e.is_forall():
            binder = "∀"
        elif e.is_exists():
            binder = "∃"
        else:
            binder = "λ"
        return binder + vs_str + ", " + printer(body, prec)
    elif smt.is_and(e):
    elif smt.is_or(e):
    elif smt.is_not(e):
    elif smt.is_app(e):
        func, args = e.decl(), e.children()
        if func in print_table:
            return print_table[func](e, prec)
        else:
            args_str = [printer(arg, prec) for arg in args]
            return str(func) + " ".join(args_str)
    



In [ ]:
{}

In [ ]:
base_derive = {
    cos : deriv_cos,
    sin : deriv_sin,
}

def derive(f, x) -> tuple[smt.ExprRef, Proof]:
    if smt.is_add(f):
        deriv_add()
    elif smt.is_mul(f):
        deriv_mul()
    else:
        d = base_derive.get(f)
        if d is not None:
            d()
        else:
            raise ValueError(f"Cannot derive function: {f}")

def cont_tac():


def lim_tac():

def prop_fun(p):
    if 

well formed term judgement
Is it meaningful to have this?
What is the interpretation? Is it a function? I'm internalizing an LCF function into a judgement?

Should I disallow 

How do you introduce a WFTerm ?

x,y,hyp |- t, Prop   is kind of a refinement

x,y,hyp |- t, p,q  ~~~ forall x y, hyp => exists t, p /\ q

Existential internalizes the _right_ of substitutions ?  Internalizes how?
  |- t, p
--------------
 exists t, p |- 

or 
|- t, p
--------
|- exists x, p(x)

|- exists x, p
---------------   this seems true and non lossy  
|- x, p

    x |- p
--------------   lossy
|- exists x, p

Well forall |- weak |- exists or something right? Adjunction?


exists_f  ?



In [ ]:
import dataclasses import dataclass
import z3

@dataclass
class Term:
    ctx : tuple[CtxItem, ...]
    term : ExprRef

# or maybe Term should be Function like in myt post?
class Function:
    # ??? preconds?
    f : z3.FuncDeclRef




In [ ]:
class Ctx(Judgement):
    ctx : tuple[CtxItem, ...]
    

In [ ]:
from z3 import *
from dataclasses import dataclass

@dataclass
class Var: # Term?
    vs : tuple[ExprRef, ...]
    def __eq__(self, other):
        if not isinstance(other, Var):
            return False
        if len(self.vs) != len(other.vs):
            return False
        return all(v1.eq(v2) for v1, v2 in zip(self.vs, other.vs))

class Term: ...

@dataclass
class Hyp: # Prop?
    fm : BoolRef

type CtxItem = Var | Hyp

@dataclass
class Proof:
    ctx : tuple[CtxItem, ...]
    conc : BoolRef  # CtxIitem?  Term | Prop

    def forall(self):
        assert isinstance(self.ctx[-1], Var)
        return Proof(ctx=self.ctx[:-1], conc=ForAll(self.ctx[-1].vs, self.conc))
    def open_forall(self):
        assert isinstance(self.conc, QuantifierRef) and self.conc.is_forall()
        vs, body = open_binder(self.conc)
        return Proof(ctx=self.ctx + (Var(vs),), conc=body)
    def implies(self, hyp: BoolRef):
        assert isinstance(self.ctx[-1], Hyp)
        return Proof(ctx=self.ctx[:-1], conc=Implies(hyp, self.conc))
    def open_implies(self):
        assert isinstance(self.conc, BoolRef) and self.conc.decl().kind() == Z3_OP_IMPLIES
        hyp, body = self.conc.children()
        return Proof(ctx=self.ctx + (Hyp(hyp),), conc=body)
    def open(self):
        if isinstance(self.conc, QuantifierRef) and self.conc.is_forall():
            return self.open_forall()
        elif isinstance(self.conc, BoolRef) and self.conc.decl().kind() == Z3_OP_IMPLIES:
            return self.open_implies()
        else:
            raise ValueError("Cannot open the current conclusion")
    def close(self):
        if isinstance(self.ctx[-1], Var):
            return self.forall()
        elif isinstance(self.ctx[-1], Hyp):
            return self.implies(self.ctx[-1].fm)
        else:
            raise ValueError("Cannot close the current context item")

    # __enter__ but it's not mutational so it makes no sense? Open a binder and then close it. Maybe for the producing zipper?
    def weaken(self, thin : list[CtxItem | None]): # Or should I give it all. Take in Ctx and reconcile automatically?
        newctx = []
        ctx = self.ctx
        pos = 0
        for t in thin:
            if t is None:
                ctx.append(self.ctx[pos])
                pos += 1
            else:
                # Check for freeness?
                match t:
                    case Var(vs):
                        assert all(free_in(self.conc, v) for v in vs)
                        for s in ctx[pos:]:
                            match s:
                                case Var(vs2):
                                    assert all(not v1.eq(v2) for v1 in vs for v2 in vs2)
                                case Hyp(fm2):
                                    ... 
                        
                    case Hyp(fm): ... # Check that we're not campturing somerhing in here in prefix?

                newctx.append(t)
        return Proof(ctx=tuple(newctx), conc=self.conc)
    def perm(self, order: list[int]):
        assert sorted(order) == range(len(self.ctx))
        newctx = tuple(self.ctx[i] for i in order)
        return Proof(ctx=newctx, conc=self.conc)


@dataclass
class MetaVars:
    vs : tuple[ExprRef, ...]
class Goal:
    ctx : tuple[CtxItem | MetaVar, ...]
    conc : BoolRef
    # allow mutation?


def axiom(ctx : tuple[CtxItem, ...], conc : BoolRef) -> Proof:
    return Proof(ctx=ctx, conc=conc)

# We don't need to quantifier over ctx
# maybe for user convenience, Ctx can take in concstant which it auto converts to Var Hyp/Prop
def prove(ctx : Ctx, conc : BoolRef, by=[]) -> Proof: 
    s = Solver()
    for h in ctx:
        match h:
            case Hyp(fm):
                s.add(fm)
            case Var(vs):
                continue
            case _:
                raise ValueError(f"Unexpected context item: {h}")
    for l in by:
        assert isinstance(l, Proof)
        l.weaken(ctx) # show that it can be weakened into this
        s.add(l.conc)
    s.add(Not(conc))
    s.check()



@dataclass
class Var:
    vs : tuple[ExprRef, ...]

@dataclass
class Prop:
    fm : BoolRef

type JudgeItem = Var | Hyp

# forest term.
@dataclass
class WFSubst:
    ctx : tuple[JudgeItem, ...]
    subst : tuple[JudgeItem, ...]
    def __matmul__(self, other: "WFSubst") -> "WFSubst": ...



In [ ]:
class Var:
    v : smt.ExprRef # vs ?

class Hyp:
    h : smt.BoolRef

class MetaVar:
    v : smt.ExprRef


class Goal:
    ctx : list[Var | Hyp | MetaVar]
    conc : smt.BoolRef


def formula_of(ctx, conc):
    f = conc
    for item in reversed(ctx):
        if isinstance(item, Hyp):
            f = smt.Implies(item.h, f)
        elif isinstance(item, Var):
            f = smt.ForAll(item.v, f)
        elif isinstance(item, MetaVar):
            f = smt.Exists(item.v, f)
    return f
def prove(ctx, conc):
    blahblahblah
    s.add(formula_of(ctx, conc))

class Proof:
    ctx : list[Var | Hyp]
    conc : smt.BoolRef


    def weaken(self, vs):
        smt.is_free_in(self.conc)
    def assume(self, hyp): # always allowed
        Proof(self.ctx + (hyp,), conc=self.conc)
    def forallI(self):
        assert isinstance(self.ctx[-1], Var)
        v = self.ctx.pop()
        return Proof(ctx=self.ctx, conc=smt.ForAll(v.v, self.conc))
    def forallE(self, t : smt.ExprRef): # instan




In [ ]:
@dataclass
class ProofZipper:
    ctx : ProofZipper | None
    goal : Goal
    left : list[Goal | Proof] # if we force left to right this would be just Proof
    right : list[Goal | Proof]
    #children : list[Goal | Proof | None] # None to mark a hole

class TopGoal(ProofZipper):
    # ctx = None
class Intro(ProofZipper):


class 

class ProofState:
    def __init__(self, goal):
        self.ctx = None
        self.goal = goal
    def intros(self):
        ...








In [6]:
x = ...

In [7]:
x is ...

True

In [ ]:
def papply(f, *args):
    if len(arg) == 0:
        return f
    else:
        papply(f(arg[0]), *args[1:])
    if x is ...:





In [33]:
type(...)

ellipsis

In [9]:
prove(cont_at(Lambda([x], x), 0), unfold=True)

|= cont_at(Lambda(x, x), 0)

In [1]:
from kdrag.theories.real import *
from kdrag.solvers import VampireSolver
from kdrag import *
x, y,a,b = Reals("x y a b")
prove(Implies(x > 0, x + 1 > 0), solver=VampireSolver)
prove(Lambda([x], RealVal(0)) != Lambda([x], x), solver=VampireSolver)


has_lim_at(Lambda([x], sin(x)), 0, 0)

@Theorem(ForAll([a,y], cont_at(Lambda(x, a), y)))
def cont_at_constant(l):
    l.unfold(cont_at).unfold(abs).auto()
#prove(ForAll(a, cont_at(Lambda(x, a), 0)), unfold=True, solver="vampire")

prove(ForAll(a, cont_at(Lambda(x, a), 0)), by=[abs.defn, cont_at.defn])

#prove(ForAll(a, cont_at(Lambda(x, a), 0)), by=[cont_at.defn], solver="vampire")
#prove(ForAll([a], cont_at(Lambda([x], RealVal(a)), 0)), unfold=True)


|= ForAll(a, cont_at(Lambda(x, a), 0))

More kdrag.theories.real to kdrag.real. I'm tired of typing that shit

lean(ForAll(a, foo(x)), by="grind") just a paste in proof scipt with mathlib avaiable. That would enable us to ground ourselves to lean and mathlib.

Could be ask the AI to see if there is an easy way to improve z3py perf

typ theory as a theory

a simple type theory prover
def has_type()

G groups
subgroup() + operator. But then why not be in set theory?

Ask for is_ground


In [54]:
from kdrag.all import *
ForAll

<function kdrag.smt.ForAll(vs: list[z3.z3.ExprRef] | z3.z3.ExprRef, *hyp_conc, **kwargs) -> z3.z3.QuantifierRef>

In [52]:
from kdrag import *
from kdrag.smt import *
from kdrag import FreshVars, Theorem, prove
from kdrag.smt import And, Consts, ForAll, Implies, Lambda, RealSort, Reals, Store, IntSort
from kdrag.theories.real import *
import kdrag.theories.real as real
from kdrag.theories.real.seq import *

a = Const("a", RSeq)
n = Const("n", IntSort())
y = Const("y", RealSort())
abs_lt = prove(ForAll(y, y < 0, real.abs(y) > 0), by=[real.abs.defn])

@PTheorem(ForAll([a,y], has_lim(a,y), ForAll(n, n >= 0, a[n] >= 0), y >= 0))
def lim_of_nonneg_seq(l):
    a,y = l.fixes()
    lim_ay = l.intros()
    pos = l.intros()
    l.unfold(has_lim, at=lim_ay)
    l.contra()
    l.specialize(0, real.abs(y))
    absypos = l.have(real.abs(y) > 0, by=[abs_lt(y)])
    l.specialize(0, absypos)
    N = l.obtain(0)
    l.have(a[N] >= 0, by=[real.abs.defn])
    l.specialize(1, N)



    #l.contra() # should return h2
    #h1 = l.have(ForAll(y, Or(y <= 0, ~has_lim(a, y))), by=[])
    #l.clear(-2)
    #l.specialize(h1)
    #l.unfold(has_lim, at=h1)




Next Goal:
a₀ : ℤ → ℝ
y₀ : ℝ
N₀ : ℤ
0: ∀n : ℤ, n > N₀ → |a₀[n] - y₀| < |y₀|
1: N₀ ≥ 0 → a₀[N₀] ≥ 0
2: ¬(y₀ ≥ 0)
3: |y₀| > 0
4: a₀[N₀] ≥ 0
───────────────────────────────────────
⊢ false


In [12]:
"""Continuity and limit lemmas for real functions.

Prove pointwise closure first, then derive global continuity and limit rules.
"""
from kdrag import *
from kdrag.smt import *
from kdrag import FreshVars, Theorem, prove
from kdrag.smt import And, Consts, ForAll, Implies, Lambda, RealSort, Reals, Store
from kdrag.theories.real import (
    RFun,
    abs,
    cont_at,
    fadd,
    fmul,
    has_lim_at,
    is_cont,
    min,
)

f, g = Consts("f g", RFun)
x, p, L, M = Reals("x p L M")

cont_at_of_is_cont = prove(
    ForAll([f, x], is_cont(f), cont_at(f, x)),
    by=[is_cont.defn],
)
@PTheorem(ForAll([f, x], cont_at(f, x), has_lim_at(f, x, f[x])))
def has_lim_at_of_cont_at(l):
    f,x = l.fixes()
    l.unfold(cont_at, has_lim_at)
    continuity = l.intros()
    eps = l.fix()
    l.intros()
    # Reuse continuity's delta for the punctured neighborhood.
    l.specialize(continuity, eps)
    delta = Real("delta")
    delta.assumes = delta > 0 
    y = Real("y")
    l.have(Exists(delta, ForAll(y, abs(x - y) < delta, abs(f[x] - f[y]) < eps)), by=[])
    delta = l.obtain(-1)
    l.exists(delta)
    l.auto(by=[abs.defn])




Lemma Complete! Change PTheorem to Theorem
